<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/transcription/05_Octave_Error_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =================================================================
# [Bass Separator] Integrated Setup & Initialization
# =================================================================
from google.colab import drive
import os
import sys

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 프로젝트 최신화 (Git Clone / Pull)
PROJECT_NAME = "Bass-separator"
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

if not os.path.exists(PROJECT_PATH):
    print(f"📦 Cloning repository... ({PROJECT_NAME})")
    !git clone {REPO_URL}
else:
    print(f"🔄 Updating repository... (Git Pull)")
    !cd {PROJECT_PATH} && git pull

# 3. 작업 경로 설정
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)
print(f"📂 Working Directory: {os.getcwd()}")

# 4. [src] 모듈을 이용한 환경 구축 및 데이터 로드
try:
    from src.env_setup import init_colab_env
    from src.utils import load_data_from_drive

    # A. 설치 및 환경 설정 (FFmpeg, Demucs, 호환성 해결)
    init_colab_env()

    # B. 데이터셋 로드
    MY_DRIVE_PATH = "/content/drive/MyDrive/Bass_separator/dataset"
    load_data_from_drive(MY_DRIVE_PATH)

except ImportError as e:
    print(f"⚠️ src 모듈 로드 실패: {e}")
    print("   Git Clone이 정상적으로 되었는지 확인해주세요.")

# 5. 자주 쓰는 라이브러리 임포트 (편의용)
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal
import soundfile as sf
from IPython.display import Audio, display
import subprocess
print("📚 Standard Libraries Imported.")

In [ ]:
# 분석할 오디오 파일의 경로를 입력
target_file_path = "/content/drive/MyDrive/Bass_separator/dataset/기타 베이스 분리 예제 2.wav"

# Demucs로 베이스 트랙 분리
print("🚀 Demucs 분리 시작...")

# 파일명 추출 (확장자 제외)
filename = os.path.splitext(os.path.basename(target_file_path))[0]

# Demucs 실행 (htdemucs 모델 사용)
cmd = f'demucs -n htdemucs "{target_file_path}"'
process = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=False)

# 분리된 베이스 파일 경로 자동 탐색
# Demucs 기본 출력 경로: separated/htdemucs/파일명/bass.wav
bass_stem_path = os.path.join('separated', 'htdemucs', filename, 'bass.wav')
print(f"✅ 분리 완료! 베이스 트랙 경로: {bass_stem_path}")

# 오디오 재생
print("🎧 분리된 베이스 트랙 듣기:")
display(Audio(bass_stem_path))

In [ ]:
from src.bass_transcription import detect_pitch

y, sr = librosa.load(bass_stem_path, sr=44100)

f0_clean = detect_pitch(y, sr)

In [ ]:
import numpy as np
import librosa
import pandas as pd

def clean_octave_errors(f0_array, valid_threshold=12, window_size=5):
    """
    주변 값과 비교하여 옥타브(12반음) 오류를 보정하는 함수

    Args:
        f0_array: Hz 단위의 피치 배열
        valid_threshold: 옥타브라고 판단할 범위 (보통 12반음 ± 1~2)
        window_size: 주변 맥락을 파악할 윈도우 크기
    """
    # 1. 원본 보존 및 복사
    f0_clean = f0_array.copy()

    # 2. Hz -> MIDI 노트로 변환 (계산 편의를 위해)
    # 0이나 NaN은 제외
    mask = (f0_clean > 0) & (~np.isnan(f0_clean))
    if np.sum(mask) == 0:
        return f0_clean

    midi_notes = np.zeros_like(f0_clean)
    midi_notes[mask] = librosa.hz_to_midi(f0_clean[mask])

    # 3. 이동 평균(Rolling Median)으로 '음악적 맥락(Trend)' 파악
    midi_series = pd.Series(midi_notes)
    # NaN이 아닌 구간에 대해서만 트렌드 계산
    trend = midi_series.rolling(window=window_size, center=True, min_periods=1).median().values

    # 4. 보정 로직 실행 (Iterative)
    # 마스크 된(유효한) 구간만 순회
    indices = np.where(mask)[0]

    for i in indices:
        current_note = midi_notes[i]
        local_trend = trend[i]

        # 트렌드와의 차이 계산
        diff = current_note - local_trend

        # 차이가 약 +12 (1옥타브 위) 라면? -> 12를 뺌
        if 10 <= diff <= 14:
            midi_notes[i] -= 12

        # 차이가 약 +24 (2옥타브 위) 라면? -> 24를 뺌 (가끔 발생)
        elif 22 <= diff <= 26:
            midi_notes[i] -= 24

        # 차이가 약 -12 (1옥타브 아래) 라면? -> 12를 더함
        elif -14 <= diff <= -10:
            midi_notes[i] += 12

    # 5. MIDI -> Hz 로 다시 변환해서 반환
    f0_clean[mask] = librosa.midi_to_hz(midi_notes[mask])

    return f0_clean